# Clinical Data Integration: SDTM & Epistemic DataFrames

This notebook demonstrates importing and analyzing clinical trial data with epistemic uncertainty:
- Loading SDTM (Study Data Tabulation Model) domains
- Creating EpistemicDataFrames with Knowledge-valued columns
- Propagating uncertainty through statistical summaries

In [ ]:
import sys
sys.path.insert(0, '../sounio-py/python')

from sounio.knowledge import Knowledge
from sounio.types import PatientData

# Try to import pandas integration
try:
    import pandas as pd
    HAS_PANDAS = True
except ImportError:
    HAS_PANDAS = False
    print("NOTE: pandas not installed. Code examples shown but cannot execute.")
    print("Install with: pip install pandas")

if HAS_PANDAS:
    print("pandas loaded successfully")

## Part 1: SDTM Domain Model Overview

SDTM defines standard clinical trial data domains:
- **DM** (Demographics): Subject info
- **EX** (Exposure): Drug dosing
- **PC** (Pharmacokinetics): Concentration measurements
- **LB** (Laboratory): Lab values
- **AE** (Adverse Events): Safety observations

In [ ]:
# Simulate SDTM DM (Demographics) domain
dm_data = {
    "USUBJID": ["001", "002", "003", "004", "005"],
    "AGE": [45, 52, 38, 61, 48],
    "SEX": ["M", "F", "M", "F", "M"],
    "WEIGHT": [82.0, 68.0, 75.0, 65.0, 90.0],
    "HEIGHT": [180.0, 165.0, 175.0, 160.0, 185.0],
}

# Simulate SDTM EX (Exposure) domain
ex_data = {
    "USUBJID": ["001", "002", "003", "004", "005"],
    "DOSE": [500.0, 500.0, 500.0, 500.0, 500.0],
    "DOSU": ["mg", "mg", "mg", "mg", "mg"],
    "ROUTE": ["PO", "PO", "PO", "PO", "PO"],
}

# Simulate SDTM PC (Pharmacokinetics) domain
# Multiple measurements per subject at different times
pc_data = {
    "USUBJID": ["001", "001", "001", "002", "002", "002", "003", "003", "003",
                 "004", "004", "004", "005", "005", "005"],
    "ATIME": [0.5, 2.0, 8.0, 0.5, 2.0, 8.0, 0.5, 2.0, 8.0,
              0.5, 2.0, 8.0, 0.5, 2.0, 8.0],  # Hours post-dose
    "AVAL": [120.5, 95.2, 15.3, 135.0, 105.1, 18.2, 110.0, 88.5, 14.1,
             125.0, 98.0, 16.5, 118.0, 92.0, 15.8],  # Concentration ng/mL
}

if HAS_PANDAS:
    dm_df = pd.DataFrame(dm_data)
    ex_df = pd.DataFrame(ex_data)
    pc_df = pd.DataFrame(pc_data)
    
    print("DM (Demographics) Domain:")
    print(dm_df.to_string())
    print("\n" + "="*60 + "\n")
    
    print("EX (Exposure) Domain:")
    print(ex_df.to_string())
    print("\n" + "="*60 + "\n")
    
    print("PC (Pharmacokinetics) Domain (subset):")
    print(pc_df.head(9).to_string())
else:
    print("# SDTM domains (DM, EX, PC) created but require pandas for display")

## Part 2: Adding Measurement Uncertainty

In [ ]:
# Enhance DM domain with epistemic weight measurements
# In practice, uncertainty comes from scale calibration, repeated measurements, etc.

epistemic_dm = {
    "USUBJID": ["001", "002", "003", "004", "005"],
    "AGE": [45, 52, 38, 61, 48],
    "SEX": ["M", "F", "M", "F", "M"],
    # Weight with measurement uncertainty (5% CV from scale)
    "WEIGHT_K": [
        Knowledge(82.0, 4.1, "scale_calib_batch_01"),
        Knowledge(68.0, 3.4, "scale_calib_batch_01"),
        Knowledge(75.0, 3.75, "scale_calib_batch_01"),
        Knowledge(65.0, 3.25, "scale_calib_batch_01"),
        Knowledge(90.0, 4.5, "scale_calib_batch_01"),
    ],
}

# Enhance PC domain with epistemic concentration measurements
# Uncertainty from HPLC assay CV
epistemic_pc = {
    "USUBJID": ["001", "001", "001", "002", "002", "002", "003", "003", "003",
                 "004", "004", "004", "005", "005", "005"],
    "ATIME": [0.5, 2.0, 8.0, 0.5, 2.0, 8.0, 0.5, 2.0, 8.0,
              0.5, 2.0, 8.0, 0.5, 2.0, 8.0],
    # Concentration with 8% CV from HPLC assay
    "AVAL_K": [
        Knowledge(120.5, 9.64, "hplc_batch_2026_w01"),
        Knowledge(95.2, 7.62, "hplc_batch_2026_w01"),
        Knowledge(15.3, 1.22, "hplc_batch_2026_w01"),
        Knowledge(135.0, 10.8, "hplc_batch_2026_w01"),
        Knowledge(105.1, 8.41, "hplc_batch_2026_w01"),
        Knowledge(18.2, 1.46, "hplc_batch_2026_w01"),
        Knowledge(110.0, 8.8, "hplc_batch_2026_w01"),
        Knowledge(88.5, 7.08, "hplc_batch_2026_w01"),
        Knowledge(14.1, 1.13, "hplc_batch_2026_w01"),
        Knowledge(125.0, 10.0, "hplc_batch_2026_w01"),
        Knowledge(98.0, 7.84, "hplc_batch_2026_w01"),
        Knowledge(16.5, 1.32, "hplc_batch_2026_w01"),
        Knowledge(118.0, 9.44, "hplc_batch_2026_w01"),
        Knowledge(92.0, 7.36, "hplc_batch_2026_w01"),
        Knowledge(15.8, 1.26, "hplc_batch_2026_w01"),
    ],
}

if HAS_PANDAS:
    epi_dm_df = pd.DataFrame(epistemic_dm)
    epi_pc_df = pd.DataFrame(epistemic_pc)
    
    print("Epistemic DM Domain (with Weight uncertainty):")
    print()
    for idx, row in epi_dm_df.iterrows():
        w_k = row['WEIGHT_K']
        print(f"Subject {row['USUBJID']}: {row['SEX']}, Age {row['AGE']} → Weight: {w_k}")
    
    print("\n" + "="*60 + "\n")
    print("Epistemic PC Domain (Concentration measurements, first 6 records):")
    print()
    for idx in range(min(6, len(epi_pc_df))):
        row = epi_pc_df.iloc[idx]
        c_k = row['AVAL_K']
        print(f"Subject {row['USUBJID']}, Time {row['ATIME']:.1f}h: {c_k}")
else:
    print("# Epistemic DataFrames created (requires pandas for display)")

## Part 3: Derived Metrics with Uncertainty

Calculate clinical metrics (BMI, dose normalization) propagating uncertainty.

In [ ]:
# Calculate BMI with uncertainty propagation
# BMI = weight / height²

height_k = Knowledge(175.0, 2.0, "measurement_tape")  # 175 cm ± 2 cm
weight_k = Knowledge(80.0, 4.0, "scale_calib")        # 80 kg ± 4 kg

# Convert height to meters for BMI
height_m = height_k / 100.0

# BMI = weight / (height_m ^ 2)
bmi = weight_k / (height_m * height_m)

print("BMI Calculation with Uncertainty Propagation:")
print(f"Height:            {height_k}")
print(f"Weight:            {weight_k}")
print(f"BMI = W/H²:        {bmi}")
print(f"BMI (formatted):   {bmi.value:.1f} ± {bmi.epsilon:.2f} kg/m²")
print(f"Relative unc:      {bmi.relative_uncertainty:.2%}")
print()

# Dose normalization: dose per kg body weight
dose_k = Knowledge(500.0, 10.0, "scale_calib")  # 500 mg ± 10 mg
dose_normalized = dose_k / weight_k

print("Dose Normalization:")
print(f"Dose:              {dose_k}")
print(f"Weight:            {weight_k}")
print(f"Dose/kg:           {dose_normalized}")
print(f"Dose/kg (formatted): {dose_normalized.value:.3f} ± {dose_normalized.epsilon:.4f} mg/kg")
print(f"Relative unc:      {dose_normalized.relative_uncertainty:.2%}")

## Part 4: Population Summary Statistics

In [ ]:
# Aggregate patient data with uncertainty
patients = [
    PatientData(
        patient_id="001",
        weight=Knowledge(82.0, 4.1, "scale_batch_01"),
        age=45,
        dose=Knowledge(500.0, 5.0, "weighing_scale"),
    ),
    PatientData(
        patient_id="002",
        weight=Knowledge(68.0, 3.4, "scale_batch_01"),
        age=52,
        dose=Knowledge(500.0, 5.0, "weighing_scale"),
    ),
    PatientData(
        patient_id="003",
        weight=Knowledge(75.0, 3.75, "scale_batch_01"),
        age=38,
        dose=Knowledge(500.0, 5.0, "weighing_scale"),
    ),
    PatientData(
        patient_id="004",
        weight=Knowledge(65.0, 3.25, "scale_batch_01"),
        age=61,
        dose=Knowledge(500.0, 5.0, "weighing_scale"),
    ),
    PatientData(
        patient_id="005",
        weight=Knowledge(90.0, 4.5, "scale_batch_01"),
        age=48,
        dose=Knowledge(500.0, 5.0, "weighing_scale"),
    ),
]

# Calculate population statistics
weights = [p.weight for p in patients]
doses = [p.dose for p in patients]

# Mean weight with uncertainty
# For independent measurements: σ_mean = sqrt(Σ σ_i²) / n
mean_weight = sum(weights[0].value for w in weights) / len(weights)
mean_weight_unc = sum(w.epsilon**2 for w in weights)**0.5 / len(weights)
mean_weight_k = Knowledge(mean_weight, mean_weight_unc, "population_mean")

print("Population Summary (N=5):")
print(f"Mean Weight: {mean_weight_k}")
print(f"  Central:   {mean_weight_k.value:.1f} kg")
print(f"  Std Error: {mean_weight_k.epsilon:.2f} kg")
print()

# Individual dose-normalized values
print("Dose Normalization by Subject:")
for p in patients:
    dose_per_kg = p.dose / p.weight
    print(f"  {p.patient_id}: {dose_per_kg.value:.3f} ± {dose_per_kg.epsilon:.4f} mg/kg")

## Part 5: Measurement Quality Assessment

In [ ]:
# Assess measurement quality across the study
print("Measurement Quality Assessment:")
print()

# Concentration measurements
conc_values = epistemic_pc["AVAL_K"]
reliable_count = sum(1 for c in conc_values if c.is_reliable(threshold=0.10))  # 10% threshold
unreliable_count = len(conc_values) - reliable_count

print(f"Concentration Measurements (HPLC assay CV = 8%):")
print(f"  Total measurements: {len(conc_values)}")
print(f"  Reliable (≤10% rel unc): {reliable_count}")
print(f"  Flagged (>10% rel unc): {unreliable_count}")
print(f"  Quality rate: {100*reliable_count/len(conc_values):.1f}%")
print()

# Weight measurements
weight_values = epistemic_dm["WEIGHT_K"]
weight_reliable = sum(1 for w in weight_values if w.is_reliable(threshold=0.08))  # 8% threshold
print(f"Weight Measurements (scale CV = 5%):")
print(f"  Total measurements: {len(weight_values)}")
print(f"  Reliable (≤8% rel unc): {weight_reliable}")
print(f"  Quality rate: {100*weight_reliable/len(weight_values):.1f}%")

## Summary

Key integration points:
1. **SDTM domains** (DM, EX, PC) represent clinical data structure
2. **Epistemic enhancement** adds measurement uncertainty from assay CVs
3. **Derived metrics** propagate uncertainty through calculations (BMI, normalization)
4. **Quality assessment** flags unreliable measurements for decision-making

This enables epistemic computing workflows where uncertainty is tracked from data entry through final analysis and reporting.